In [1]:
import os
import sys
import json
from model.rag_4b import solve_math_direct_4b, solve_math_question_4b

d:\User\ProjectGithub\hiepnguyenn-99\RAG-Solve-Math\appenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('data/cleaned/test/dai_so.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
test_data_math = data["problems"]
print(f"Số lượng bài test: {len(test_data_math)}")

Số lượng bài test: 354


In [3]:
def solve_math_batch(questions):
    """
    Nhận list câu hỏi, trả về list đáp án bằng cách gọi solve_math_question_4b từng câu.
    Nếu gặp lỗi, trả về None cho câu đó.
    """
    answers = []
    for q in questions:
        try:
            result = solve_math_question_4b(q)
            ans = result if isinstance(result, str) else result[0]
        except Exception as e:
            ans = None
        answers.append(ans)
    return answers

In [4]:
def process_in_batches(questions, batch_size):
    """
    """
    all_answers = []
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        batch_answers = solve_math_batch(batch)
        all_answers.extend(batch_answers)
    return all_answers

TEST giải toán

In [5]:
# Test giải toán trên dữ liệu test và ghi kết quả từng câu ngay sau khi xử lý
# batch_size = 32
batch_size = 5
questions = [item['question'] for item in test_data_math[:200]]
start = 162
os.makedirs('Answer', exist_ok=True)
with open('Answer/qwen4b/dai_so_with_context.json', 'a', encoding='utf-8') as f:
    if start == 0:
        f.write('[\n')
    for idx in range(start, len(questions), batch_size):
        batch = questions[idx:idx+batch_size]
        answers = process_in_batches(batch, batch_size=len(batch))
        for i, (question, answer) in enumerate(zip(batch, answers), idx+1):
            obj = {"index": i, "input": question, "output": answer}
            if i == len(questions):
                f.write(json.dumps(obj, ensure_ascii=False) + '\n')
            else:
                f.write(json.dumps(obj, ensure_ascii=False) + ',\n')
        print(f"Đã ghi gần đến câu thứ {idx+len(batch)}")
    f.write(']\n')

Đã ghi gần đến câu thứ 167
Đã ghi gần đến câu thứ 172
Đã ghi gần đến câu thứ 177
Đã ghi gần đến câu thứ 182
Đã ghi gần đến câu thứ 187
Đã ghi gần đến câu thứ 192
Đã ghi gần đến câu thứ 197
Đã ghi gần đến câu thứ 200


In [ ]:
# Kiểm tra lại file answer RAG đã ghi ra
with open('Answer/qwen4b/dai_so.json', 'r', encoding='utf-8') as f:
    answer_rag_data = json.load(f)
print(f"Số lượng kết quả trong file Answer/qwen4b/dai_so.json RAG: {len(answer_rag_data)}")

Số lượng kết quả trong file answer RAG: 200
Ví dụ kết quả đầu tiên: {'index': 1, 'input': 'Cho ma trận $A = \\begin{pmatrix} 1 & 2 \\\\ 3 & 4 \\end{pmatrix}$ và đa thức $p(x) = x^2 - 5x - 2$. a) Tính $A^2$. b) Tính $p(A)$, tức là thay ma trận $A$ vào đa thức $p(x)$. Lưu ý: hằng số tự do trong đa thức được thay bằng hằng số đó nhân với ma trận đơn vị $I$ cùng cấp. c) Nhận xét về kết quả của $p(A)$.', 'output': '1. Phân tích đề bài và xác định phương pháp giải  \n- Ta có ma trận $ A = \\begin{pmatrix} 1 & 2 \\\\ 3 & 4 \\end{pmatrix} $  \n- Đa thức $ p(x) = x^2 - 5x - 2 $  \n- Câu a): Tính $ A^2 $: nhân ma trận $ A $ với chính nó  \n- Câu b): Tính $ p(A) = A^2 - 5A - 2I $, với $ I $ là ma trận đơn vị cấp 2  \n- Câu c): Nhận xét về kết quả $ p(A) $\n\n---\n\n2. Giải chi tiết từng bước  \n\na) Tính $ A^2 $  \n$$\nA^2 = A \\cdot A = \\begin{pmatrix} 1 & 2 \\\\ 3 & 4 \\end{pmatrix} \\begin{pmatrix} 1 & 2 \\\\ 3 & 4 \\end{pmatrix}\n$$  \nTính từng phần:  \n- Phần (1,1): $ 1\\cdot1 + 2\\cdot3 =